<a href="https://colab.research.google.com/github/GaganDeshbhandari/ml-project/blob/main/scripts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**READ EVERY TEXTS BELOW THE CELL**




In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Drive is now linked to this notebook can now access the drive to get the datasets


**NOTE : Always run the above cell before running any other cell because we need to link the Datasets from the Drive**

In [2]:
import pandas as pd
import os
import random

All the Libraries required for the project are imported in the above cell

In [3]:
BASE_PATH = '/content/drive/MyDrive/mini-project'
CLAIMS_DATA_PATH = os.path.join(BASE_PATH,'datasets/claims_datasets.csv')
USER_PERCEPTION_DATA = os.path.join(BASE_PATH,'datasets/user_perception_dataset.csv')

Paths for the datasets

In [4]:
claims_df = pd.read_csv(CLAIMS_DATA_PATH)
# display(claims_df)

Above cell displays the claims after reading from the Drive.

If its showing error make sure that you have executed the Drive Mounting and Library,Package importing cell

In [ ]:

# claims = list(range(1, 32))  # claim_id 1 to 31
# users = [f"U{i}" for i in range(1, 21)]

# ratings_by_truth = {
#     "True": ["True"]*10 + ["Mostly True"]*7 + ["Mixture"]*3,
#     "Mostly True": ["Mostly True"]*9 + ["True"]*5 + ["Mixture"]*4 + ["Mostly False"]*2,
#     "Mixture": ["Mixture"]*8 + ["Mostly True"]*4 + ["Mostly False"]*4 + ["True"]*2 + ["False"]*2,
#     "Mostly False": ["Mostly False"]*9 + ["False"]*6 + ["Mixture"]*3 + ["Mostly True"]*2,
#     "False": ["False"]*10 + ["Mostly False"]*7 + ["Mixture"]*3
# }

# claims_df = pd.read_csv(CLAIMS_DATA_PATH)

# rows = []

# for _, row in claims_df.iterrows():
#     truth = row["ground_truth"]
#     possible_ratings = ratings_by_truth[truth]

#     for user in users:
#         rows.append({
#             "claim_id": row["claim_id"],
#             "user_id": user,
#             "rating": random.choice(possible_ratings)
#         })

# perception_df = pd.DataFrame(rows)

# perception_df.to_csv(USER_PERCEPTION_DATA, index=False)

# print("perception_dataset.csv created")
# print(perception_df)

perception_dataset.csv created
      claim_id user_id        rating
0            1      U1          True
1            1      U2          True
2            1      U3   Mostly True
3            1      U4   Mostly True
4            1      U5   Mostly True
...        ...     ...           ...
2415       121     U16         False
2416       121     U17  Mostly False
2417       121     U18       Mixture
2418       121     U19  Mostly False
2419       121     U20  Mostly False

[2420 rows x 3 columns]


The above cell generates the Random User Perception Data based on the ground truth of the Claims.

So that if the Claim is True most of the users will think that it is actually True.
And if the Claim is False most of the users will think that it is actually False.

And if the Claim is Mixture or Mostly True or Mostly False user perception changes randomly


**NOTE :** **THE ABOVE CELL IS EXECUTED ONLY ONCE BECAUSE IF EXECUTED ONCE MORE THE USER_PERCEPTION_DATA GETS REPLACED.**

**THATS WHY I HAVE COMMENTED IT**

In [5]:
user_perception_df = pd.read_csv(USER_PERCEPTION_DATA)
# display(user_perception_df)

The above cell displays the User_perception_data

In [6]:
gtl_mapping = {
    "True": 1.0,
    "Mostly True": 0.5,
    "Mixture": 0,
    "Mostly False": -0.5,
    "False": -1
}
# GTL = Ground Truth Level
claims_df['GTL'] = claims_df['ground_truth'].map(gtl_mapping)
print(claims_df.head())

   claim_id                                              claim ground_truth  \
0         1  India expanded 5G services to more tier-2 citi...         True   
1         2  UPI transactions reached record high monthly v...         True   
2         3  ISRO conducted successful test of reusable lau...         True   
3         4  Government extended free food grain scheme for...         True   
4         5  Digital payments usage increased significantly...         True   

           source  GTL  
0  PIB Fact Check  1.0  
1  PIB Fact Check  1.0  
2  PIB Fact Check  1.0  
3  PIB Fact Check  1.0  
4       BOOM Live  1.0  


The above cell maps the ground_truth to the GTL

In [7]:
rating_mapping = {
    "True": 1.0,
    "Mostly True": 0.5,
    "Mixture": 0,
    "Mostly False": -0.5,
    "False": -1
}
# PTL = Perceived Truth Level
user_perception_df['PTL'] = user_perception_df['rating'].map(rating_mapping)
print(user_perception_df)

      claim_id user_id        rating  PTL
0            1      U1          True  1.0
1            1      U2          True  1.0
2            1      U3   Mostly True  0.5
3            1      U4   Mostly True  0.5
4            1      U5   Mostly True  0.5
...        ...     ...           ...  ...
3441       171     U16         False -1.0
3442       171     U17  Mostly False -0.5
3443       171     U18  Mostly False -0.5
3444       171     U19         False -1.0
3445       171     U20  Mostly False -0.5

[3446 rows x 4 columns]


The above cell maps the user_perception to the PTL

In [8]:
PTL_per_claim = user_perception_df.groupby('claim_id')['PTL'].mean().reset_index()
PTL_per_claim.columns = ['claim_id', 'PTL']
# display(PTL_per_claim)

The above cell calculates the PTL of each claim

In [9]:
merged_df = pd.merge(PTL_per_claim, claims_df[['claim_id', 'GTL']], on='claim_id')
# display(merged_df)

Above cell merges the PTL per claim and GTL as one table

In [10]:
merged_df['TPB'] = abs(merged_df['PTL'] - merged_df['GTL'])
merged_df['TPB_label'] = merged_df['TPB'].apply(lambda x: 'High' if x >= merged_df['TPB'].median() else 'Low')
# display(merged_df)

Above cell calculates the TPB = Total Perception Bias of the training datasets and provides the label for the TPB

and marks the TPB_label when TPB is greater than the median of TPB

In [11]:
from scipy.stats import skew

features_df = user_perception_df.groupby('claim_id')['PTL'].agg(
    mean='mean',
    median='median',
    variance='var',
    skewness=lambda x: skew(x)
).reset_index()

# display(features_df)

/tmp/ipykernel_1476/2659704560.py:7: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  skewness=lambda x: skew(x)


The above cell calculates mean, median, variance, skewness of the user_perception based on each claim

MEAN :  sum of all values / total count

MEDIAN : middle value after sorting

---

VARIANCE : how spread out the ratings are


If all users said True → variance = 0 (everyone agreed)


If users said mix of True and False → variance is high (everyone disagreed)


For claim 1 → variance is low because most users agreed it was True/Mostly True

---
SKEWNESS : which side the ratings are leaning towards


Positive skew → most ratings leaning towards False side


Negative skew → most ratings leaning towards True side


Zero → ratings evenly spread


For claim 1 → negative skew because most users rated True/Mostly True

In [12]:
final_df = pd.merge(features_df, merged_df[['claim_id', 'GTL', 'TPB', 'TPB_label']], on='claim_id')
# display(final_df)

The above cell marks the final dataframe

and displays all the features of the datasets after calculating it

In [13]:
X = final_df[['mean', 'median', 'variance', 'skewness']]
y = final_df['TPB_label']

X indicates question that is given to the model

Y indicates the output that is given by the model

During training the model looks at X and y together and learns the pattern — like "when variance is high and mean is low, TPB is usually High".

In [15]:
X = X.fillna(0)

In [21]:
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

models = {
    'Linear SVM': LinearSVC(),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(),
    'Random Forest': RandomForestClassifier()
}

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')
    print(f"{name}: {scores.mean():.2f} ± {scores.std():.2f}")

Linear SVM: 0.51 ± 0.10
Naive Bayes: 0.70 ± 0.12
Logistic Regression: 0.53 ± 0.07
Random Forest: 0.92 ± 0.08


# Model Selection — Why Random Forest?

We trained four classifiers to predict TPB (Total Perception Bias) and compared their accuracies:

| Model | Accuracy |
|---|---|
| Linear SVM | 51% |
| Naive Bayes | 70% |
| Logistic Regression | 53% |
| **Random Forest** | **91%** |

---

## Why the others failed

- **Linear SVM & Logistic Regression** — Both try to separate High/Low TPB using a straight line. Since the relationship between user perception features and TPB is complex and non-linear, a straight line is not enough.

- **Naive Bayes** — Assumes all features (mean, variance, skewness) are completely independent of each other. In reality they are related, which causes incorrect predictions.

---

## Why Random Forest wins

Random Forest builds hundreds of decision trees, each asking simple questions about the features:

```
Is mean > 0.3?
    Yes → Is variance < 0.1? → High TPB
    No  → Low TPB
```

By combining results from all trees it captures **complex non-linear patterns** in the data that the other models miss.

> ✅ **Random Forest was selected as the final model with 91% accuracy.**

In [22]:
import joblib

from sklearn.ensemble import RandomForestClassifier

best_model = RandomForestClassifier()
best_model.fit(X_scaled, y)

joblib.dump(best_model, '/content/drive/MyDrive/mini-project/random_forest_model.pkl')
joblib.dump(scaler, '/content/drive/MyDrive/mini-project/scaler.pkl')

print("Model saved successfully!")

Model saved successfully!


# Saving the Model

After training, we save the best model (Random Forest) to Google Drive so it can be reused later without retraining.

## What gets saved

| File | Purpose |
|---|---|
| `random_forest_model.pkl` | The trained Random Forest model |
| `scaler.pkl` | The StandardScaler used to scale features |

> ⚠️ Both files must be saved together — the scaler is needed to process new data before feeding it to the model.